# ⚙️ 04 — Feature Engineering
**Startup Funding Analysis Project**  
This notebook transforms the cleaned raw data into a rich, ML-ready feature matrix.  
Steps: Temporal features, Investor Network Graph (PageRank), Company-level aggregations, Categorical Encoding, and Target Variable creation.

## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import itertools
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

warnings.filterwarnings('ignore')

plt.rcParams['figure.facecolor'] = '#0f0f1a'
plt.rcParams['axes.facecolor'] = '#1a1a2e'
plt.rcParams['axes.edgecolor'] = '#3a3a5c'
plt.rcParams['axes.labelcolor'] = '#c0c0e0'
plt.rcParams['xtick.color'] = '#a0a0c0'
plt.rcParams['ytick.color'] = '#a0a0c0'
plt.rcParams['text.color'] = '#e0e0f0'
plt.rcParams['grid.color'] = '#2a2a4a'

PALETTE = ['#5B8DEF', '#8E5BEF', '#EF5B8D', '#EFB85B', '#5BEFB8', '#EF8E5B']
os.makedirs('../visualizations', exist_ok=True)
os.makedirs('../reports', exist_ok=True)

print('✅ Libraries loaded successfully!')

## 2. Load Cleaned Raw Dataset

In [ ]:
input_path = '../data/raw/Startup_Funding_Cleaned.csv'
df = pd.read_csv(input_path, low_memory=False)

# Parse date if not already done
if 'funded_at' in df.columns:
    df['funded_at'] = pd.to_datetime(df['funded_at'], errors='coerce')
    df['funding_year'] = df['funded_at'].dt.year
    df['funding_month'] = df['funded_at'].dt.month

print(f'✅ Dataset loaded successfully!')
print(f'   Rows    : {df.shape[0]:,}')
print(f'   Columns : {df.shape[1]}')
df.head(3)

## 3. Temporal & Macro Features

In [ ]:
# --- Feature 1: Recession Era Impact (2008-09 Global Financial Crisis) ---
df['is_recession_era'] = df['funding_year'].isin([2008, 2009]).astype(int)

# --- Feature 2: Tech Boom Era (2014-2021 peak VC years) ---
df['is_tech_boom'] = df['funding_year'].isin(range(2014, 2022)).astype(int)

# --- Feature 3: Post-COVID Era (2021+) ---
df['is_post_covid'] = (df['funding_year'] >= 2021).astype(int)

# --- Feature 4: Funding Seasonality (Q4 = higher funding activity) ---
if 'funding_month' in df.columns:
    df['funding_quarter'] = df['funded_at'].dt.quarter
    df['is_q4_funding'] = (df['funding_quarter'] == 4).astype(int)

print('✅ Temporal & Macro features created!')
print(f"Recession-era records : {df['is_recession_era'].sum():,}")
print(f"Tech-boom era records : {df['is_tech_boom'].sum():,}")
print(f"Post-COVID records    : {df['is_post_covid'].sum():,}")

## 4. Build Co-Investment Network Graph

In [ ]:
print('Building co-investment network graph...')
print('This may take 1-3 minutes for large datasets...\n')

# Filter to known investors only
network_data = df[df['Investor_Name'].notna()].copy()
network_data = network_data[
    ~network_data['Investor_Name'].astype(str).str.lower().str.contains(
        'undisclosed|unnamed|unknown|n/a', na=False
    )
]

# Group investors per funding round
round_groups = network_data.groupby('funding_round_id')['Investor_Name'].apply(list)

# Build all pairwise co-investment edges
co_investment_pairs = []
for investors in round_groups:
    if len(investors) > 1:
        unique_investors = sorted(list(set(str(i) for i in investors)))
        for pair in itertools.combinations(unique_investors, 2):
            co_investment_pairs.append(pair)

# Build weighted edge list
edges_df = pd.DataFrame(co_investment_pairs, columns=['Source', 'Target'])
edges_df = edges_df.groupby(['Source', 'Target']).size().reset_index(name='Weight')

print(f'✅ Network graph edges built!')
print(f'   Total co-investment pairs: {len(edges_df):,}')
print(f'   Unique investor nodes    : {len(set(edges_df["Source"]).union(set(edges_df["Target"]))):,}')

## 5. Calculate PageRank & Network Centrality Metrics

In [ ]:
# Build NetworkX graph
G = nx.from_pandas_edgelist(
    edges_df, source='Source', target='Target',
    edge_attr='Weight', create_using=nx.Graph()
)

print(f'Graph: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges')

# Compute PageRank (influence score)
pagerank_scores = nx.pagerank(G, weight='Weight', alpha=0.85, max_iter=200)

# Compute Degree Centrality
degree_centrality = nx.degree_centrality(G)

# Compute Betweenness for top 500 nodes only (computationally expensive)
print('Computing betweenness centrality for top 500 nodes...')
top_500_nodes = sorted(pagerank_scores, key=pagerank_scores.get, reverse=True)[:500]
sub_G = G.subgraph(top_500_nodes)
betweenness = nx.betweenness_centrality(sub_G, normalized=True)

# Median PageRank as fallback for unknown investors
median_pagerank = np.median(list(pagerank_scores.values()))

print(f'\n✅ PageRank computation complete!')
print(f'   Median PageRank : {median_pagerank:.8f}')
top5 = sorted(pagerank_scores.items(), key=lambda x: x[1], reverse=True)[:5]
print('\n   Top 5 Most Influential Investors:')
for i, (name, score) in enumerate(top5, 1):
    print(f'   {i}. {name:40s} → PageRank: {score:.8f}')

## 6. Save Investor Network Files

In [ ]:
os.makedirs('../data/processed', exist_ok=True)

# 1. Save network edges
edges_path = '../data/processed/investor_network_edges.csv'
edges_df.to_csv(edges_path, index=False)
print(f'✅ Network edges saved: {edges_path}')

# 2. Build and save centrality data
centrality_records = []
for node in G.nodes():
    centrality_records.append({
        'Investor_Name': node,
        'PageRank_Centrality': pagerank_scores.get(node, 0),
        'Degree_Centrality': degree_centrality.get(node, 0),
        'Degree': G.degree(node),
        'Betweenness_Centrality': betweenness.get(node, 0)
    })

centrality_df = pd.DataFrame(centrality_records).sort_values('PageRank_Centrality', ascending=False)
centrality_path = '../data/processed/investor_centrality.csv'
centrality_df.to_csv(centrality_path, index=False)
print(f'✅ Centrality data saved: {centrality_path}')
print(f'   Total investors tracked: {len(centrality_df):,}')
centrality_df.head(10)

## 7. Map Investor Centrality Back to Companies

In [ ]:
def get_investor_centrality(investor_list):
    """Calculate max and sum PageRank for a company's investors."""
    if not isinstance(investor_list, list):
        return 0.0, 0.0, 0
    valid_pagers = []
    for inv in investor_list:
        inv_str = str(inv)
        if pd.isna(inv) or any(kw in inv_str.lower() for kw in ['undisclosed', 'unnamed', 'unknown']):
            continue
        valid_pagers.append(pagerank_scores.get(inv_str, median_pagerank))
    if not valid_pagers:
        return 0.0, 0.0, 0
    return max(valid_pagers), sum(valid_pagers), len(valid_pagers)


# Group all investors per company
company_investors = df.groupby('company_id')['Investor_Name'].apply(
    lambda x: list(set(str(i) for i in x if pd.notna(i)))
).reset_index()

# Apply centrality function
print('Mapping centrality scores to companies...')
company_investors[['Max_Investor_PageRank', 'Sum_Investor_PageRank', 'Unique_Investors_Count']] = \
    company_investors['Investor_Name'].apply(
        lambda x: pd.Series(get_investor_centrality(x))
    )

print(f'✅ Centrality mapped for {len(company_investors):,} companies!')
print(f'   Companies with known investors: {(company_investors["Unique_Investors_Count"] > 0).sum():,}')
company_investors.head(5)

## 8. Company-Level Aggregation (Transaction → Profile)

In [ ]:
print('Aggregating transaction-level data to company level...')

company_features = df.groupby('company_id').agg(
    # Funding metrics
    Total_Funding_USD=('raised_amount_usd', 'sum'),
    Max_Round_Funding=('raised_amount_usd', 'max'),
    Avg_Round_Funding=('raised_amount_usd', 'mean'),
    Median_Round_Funding=('raised_amount_usd', 'median'),
    # Round metrics
    Total_Funding_Rounds=('funding_round_id', 'nunique'),
    # Timeline
    First_Funding_Year=('funding_year', 'min'),
    Last_Funding_Year=('funding_year', 'max'),
    # Macro conditions
    Fought_Through_Recession=('is_recession_era', 'max'),
    Funded_During_Tech_Boom=('is_tech_boom', 'max'),
    Funded_Post_COVID=('is_post_covid', 'max'),
    Has_Q4_Funding=('is_q4_funding', 'max'),
).reset_index()

# Derived features
company_features['Age_at_Latest_Round'] = (
    company_features['Last_Funding_Year'] - company_features['First_Funding_Year']
) + 1

company_features['Log_Total_Funding'] = np.log1p(company_features['Total_Funding_USD'])
company_features['Log_Max_Round_Funding'] = np.log1p(company_features['Max_Round_Funding'])

# Funding velocity: Total USD per year of activity
company_features['Funding_Velocity'] = (
    company_features['Total_Funding_USD'] / company_features['Age_at_Latest_Round']
).replace([np.inf, -np.inf], 0).fillna(0)

print(f'✅ Company-level aggregation complete!')
print(f'   Total unique companies: {len(company_features):,}')
print(f'   New features created  : {len(company_features.columns)}')
company_features.head(3)

## 9. Merge Network Features into Company Matrix

In [ ]:
# Merge PageRank centrality features
company_features = pd.merge(
    company_features,
    company_investors[['company_id', 'Max_Investor_PageRank', 'Sum_Investor_PageRank', 'Unique_Investors_Count']],
    on='company_id',
    how='left'
)

# Fill NaN centrality for companies with no known investors
company_features['Max_Investor_PageRank'] = company_features['Max_Investor_PageRank'].fillna(0)
company_features['Sum_Investor_PageRank'] = company_features['Sum_Investor_PageRank'].fillna(0)
company_features['Unique_Investors_Count'] = company_features['Unique_Investors_Count'].fillna(0)

# Log-transform PageRank sum (heavy right skew)
company_features['Log_Sum_Investor_PageRank'] = np.log1p(company_features['Sum_Investor_PageRank'] * 1e6)

print(f'✅ Network features merged!')
print(f'   Companies with investor data: {(company_features["Unique_Investors_Count"] > 0).sum():,}')
print(f'   Feature matrix shape: {company_features.shape}')

## 10. Merge Company Static Profile

In [ ]:
# Extract static company metadata (one row per company)
profile_cols = [c for c in ['company_id', 'Startup_Name', 'Industry_Sector', 'country_code', 'state_code', 'city', 'Startup_Status'] if c in df.columns]
company_profile = df.drop_duplicates(subset=['company_id'], keep='last')[profile_cols]

# Merge with aggregated features
final_df = pd.merge(company_features, company_profile, on='company_id', how='left')

print(f'✅ Profile merged!')
print(f'   Final matrix shape: {final_df.shape}')
print(f'   Columns: {list(final_df.columns)}')
final_df.head(3)

## 11. Create Target Variable

In [ ]:
# Binary target: 1 = successful exit (Acquired or IPO), 0 = still operating/closed/unknown
if 'Startup_Status' in final_df.columns:
    final_df['is_successful'] = final_df['Startup_Status'].apply(
        lambda x: 1 if str(x).strip().lower() in ['acquired', 'ipo'] else 0
    )

    success_count = final_df['is_successful'].sum()
    total_count = len(final_df)
    print(f'✅ Target variable created!')
    print(f'   Successful exits (1) : {success_count:,}  ({success_count/total_count*100:.2f}%)')
    print(f'   Non-successful   (0) : {total_count - success_count:,}  ({(total_count-success_count)/total_count*100:.2f}%)')
    print(f'   Class imbalance ratio: {(total_count - success_count)/success_count:.1f}:1 (negative:positive)')
else:
    print('⚠️  Startup_Status column not found. Setting is_successful = 0 as fallback.')
    final_df['is_successful'] = 0

## 12. Categorical Cleaning & Grouping

In [ ]:
# Group low-frequency countries into 'Other' (keep top 10)
top_countries = final_df['country_code'].value_counts().head(10).index.tolist()
final_df['country_code_cleaned'] = final_df['country_code'].apply(
    lambda x: x if x in top_countries else 'Other'
)

# Group low-frequency industry sectors (keep top 20, rest = 'other')
final_df['Industry_Sector_cleaned'] = final_df['Industry_Sector'].astype(str).str.strip().str.lower()
top_sectors = final_df['Industry_Sector_cleaned'].value_counts().head(20).index.tolist()
final_df['Industry_Sector_cleaned'] = final_df['Industry_Sector_cleaned'].apply(
    lambda x: x if x in top_sectors else 'other'
)

print('✅ Categorical columns cleaned and grouped!')
print(f'   Unique country_code_cleaned values : {final_df["country_code_cleaned"].nunique()}')
print(f'   Unique Industry_Sector_cleaned values: {final_df["Industry_Sector_cleaned"].nunique()}')
print(f'\n   Country distribution:')
print(final_df['country_code_cleaned'].value_counts().to_string())
print(f'\n   Top 10 Sector distribution:')
print(final_df['Industry_Sector_cleaned'].value_counts().head(10).to_string())

## 13. Feature Distribution Analysis

In [ ]:
numeric_features = [
    'Total_Funding_Rounds', 'Unique_Investors_Count', 'Age_at_Latest_Round',
    'Log_Total_Funding', 'Max_Investor_PageRank', 'Funding_Velocity'
]
numeric_features = [f for f in numeric_features if f in final_df.columns]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, feat in enumerate(numeric_features):
    data = final_df[feat].dropna()
    # Clip to 99th percentile for clean visualization
    data_clipped = data.clip(upper=data.quantile(0.99))
    axes[i].hist(data_clipped, bins=50, color=PALETTE[i % len(PALETTE)], alpha=0.85, edgecolor='#3a3a5c')
    axes[i].set_title(f'{feat}', fontweight='bold', fontsize=10)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')
    axes[i].grid(alpha=0.3)
    axes[i].axvline(data.median(), color='#EF5B8D', linestyle='--', linewidth=1.5, label=f'Median: {data.median():.2f}')
    axes[i].legend(fontsize=8)

plt.suptitle('Feature Distributions (Clipped at P99)', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../visualizations/feature_distributions.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

## 14. Feature vs Success Analysis

In [ ]:
# Compare distributions between successful and non-successful startups
if 'is_successful' in final_df.columns:
    compare_features = [
        ('Total_Funding_Rounds', 'Funding Rounds'),
        ('Unique_Investors_Count', 'Investor Count'),
        ('Log_Total_Funding', 'Log Total Funding'),
        ('Max_Investor_PageRank', 'Max Investor PageRank')
    ]
    compare_features = [(f, l) for f, l in compare_features if f in final_df.columns]

    success_df = final_df[final_df['is_successful'] == 1]
    fail_df = final_df[final_df['is_successful'] == 0]

    fig, axes = plt.subplots(1, len(compare_features), figsize=(18, 5))

    for i, (feat, label) in enumerate(compare_features):
        s_data = success_df[feat].dropna().clip(upper=success_df[feat].quantile(0.99))
        f_data = fail_df[feat].dropna().clip(upper=fail_df[feat].quantile(0.99))

        axes[i].hist(s_data, bins=40, color='#5BEFB8', alpha=0.6, label='Successful', edgecolor='none')
        axes[i].hist(f_data, bins=40, color='#EF5B8D', alpha=0.5, label='Not Successful', edgecolor='none')
        axes[i].set_title(label, fontweight='bold', fontsize=10)
        axes[i].set_xlabel('Value')
        axes[i].legend(fontsize=8)
        axes[i].grid(alpha=0.3)

    plt.suptitle('Feature Distributions: Successful vs Non-Successful Startups', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../visualizations/success_vs_fail_distributions.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()

    print('\n📊 Statistical Comparison (Median values):')
    for feat, label in compare_features:
        if feat in final_df.columns:
            s_med = success_df[feat].median()
            f_med = fail_df[feat].median()
            print(f'  {label:30s}: Successful={s_med:.4f}, Non-Successful={f_med:.4f}, Ratio={s_med/f_med:.2f}x' if f_med > 0 else f'  {label}: N/A')

## 15. Feature Correlation Matrix

In [ ]:
corr_cols = [
    'Total_Funding_USD', 'Total_Funding_Rounds', 'Unique_Investors_Count',
    'Age_at_Latest_Round', 'Log_Total_Funding', 'Max_Investor_PageRank',
    'Sum_Investor_PageRank', 'Funding_Velocity', 'Fought_Through_Recession',
    'Funded_During_Tech_Boom', 'is_successful'
]
corr_cols = [c for c in corr_cols if c in final_df.columns]

corr_matrix = final_df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlBu', center=0, vmin=-1, vmax=1,
    linewidths=0.5, linecolor='#1a1a2e',
    cbar_kws={'shrink': 0.8},
    ax=ax, annot_kws={'size': 8}
)
ax.set_title('Engineered Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../visualizations/feature_correlation_matrix.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

## 16. Top Investor PageRank Visualization

In [ ]:
# Visualize top 20 investors by PageRank
top_investors_pr = sorted(pagerank_scores.items(), key=lambda x: x[1], reverse=True)[:20]
inv_names = [x[0][:30] for x in top_investors_pr]
inv_scores = [x[1] for x in top_investors_pr]

fig, ax = plt.subplots(figsize=(14, 8))
colors = plt.cm.plasma(np.linspace(0.25, 0.9, len(inv_names)))
bars = ax.barh(inv_names[::-1], inv_scores[::-1], color=colors, alpha=0.9, edgecolor='#3a3a5c')

for bar, score in zip(bars, inv_scores[::-1]):
    ax.text(bar.get_width() + 0.000001, bar.get_y() + bar.get_height() / 2,
            f'{score:.7f}', va='center', ha='left', fontsize=8, color='#c0c0e0')

ax.set_title('Top 20 Investors by PageRank Centrality (Network Influence)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('PageRank Score')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../visualizations/top_investors_pagerank.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

## 17. Funding Rounds Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Rounds histogram
round_data = final_df['Total_Funding_Rounds'].clip(upper=15)
axes[0].hist(round_data, bins=15, color=PALETTE[0], alpha=0.85, edgecolor='#3a3a5c')
axes[0].set_title('Distribution of Total Funding Rounds per Startup', fontweight='bold')
axes[0].set_xlabel('Number of Funding Rounds')
axes[0].set_ylabel('Number of Companies')
axes[0].grid(alpha=0.3)

# Success rate by rounds
if 'is_successful' in final_df.columns:
    rounds_success = final_df.groupby('Total_Funding_Rounds')['is_successful'].mean().reset_index()
    rounds_success = rounds_success[rounds_success['Total_Funding_Rounds'] <= 12]
    axes[1].plot(rounds_success['Total_Funding_Rounds'], rounds_success['is_successful'] * 100,
                 color=PALETTE[2], linewidth=2.5, marker='o', markersize=6)
    axes[1].fill_between(rounds_success['Total_Funding_Rounds'], rounds_success['is_successful'] * 100, alpha=0.2, color=PALETTE[2])
    axes[1].set_title('Success Rate (%) by Number of Funding Rounds', fontweight='bold')
    axes[1].set_xlabel('Number of Funding Rounds')
    axes[1].set_ylabel('Success Rate (%)')
    axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../visualizations/funding_rounds_analysis.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

## 18. Success Rate by Industry Sector

In [ ]:
if 'is_successful' in final_df.columns and 'Industry_Sector_cleaned' in final_df.columns:
    sector_success = final_df.groupby('Industry_Sector_cleaned').agg(
        success_rate=('is_successful', 'mean'),
        count=('is_successful', 'count')
    ).reset_index()
    # Filter sectors with at least 20 companies
    sector_success = sector_success[sector_success['count'] >= 20]
    sector_success = sector_success.sort_values('success_rate', ascending=True)

    fig, ax = plt.subplots(figsize=(14, 7))
    colors = plt.cm.RdYlGn(sector_success['success_rate'].values)
    bars = ax.barh(sector_success['Industry_Sector_cleaned'], sector_success['success_rate'] * 100,
                   color=colors, alpha=0.85, edgecolor='#3a3a5c')

    for bar, cnt in zip(bars, sector_success['count']):
        ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
                f'n={cnt:,}', va='center', ha='left', fontsize=8, color='#c0c0e0')

    ax.set_title('Success Rate (%) by Industry Sector (min. 20 companies)', fontsize=13, fontweight='bold', pad=15)
    ax.set_xlabel('Success Rate (%)')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig('../visualizations/success_rate_by_sector.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()

## 19. Success Rate by Country

In [ ]:
if 'is_successful' in final_df.columns and 'country_code_cleaned' in final_df.columns:
    country_success = final_df.groupby('country_code_cleaned').agg(
        success_rate=('is_successful', 'mean'),
        total_companies=('is_successful', 'count'),
        total_funding=('Total_Funding_USD', 'sum')
    ).reset_index()
    country_success = country_success.sort_values('success_rate', ascending=False)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Success rate
    colors = plt.cm.plasma(np.linspace(0.2, 0.9, len(country_success)))
    axes[0].bar(country_success['country_code_cleaned'], country_success['success_rate'] * 100,
                color=colors, alpha=0.9, edgecolor='#3a3a5c')
    axes[0].set_title('Success Rate (%) by Country', fontweight='bold')
    axes[0].set_xlabel('Country Code')
    axes[0].set_ylabel('Success Rate (%)')
    axes[0].tick_params(axis='x', rotation=30)
    axes[0].grid(axis='y', alpha=0.3)

    # Total funding
    axes[1].bar(country_success['country_code_cleaned'], country_success['total_funding'] / 1e9,
                color=colors, alpha=0.9, edgecolor='#3a3a5c')
    axes[1].set_title('Total Capital Deployed by Country ($B)', fontweight='bold')
    axes[1].set_xlabel('Country Code')
    axes[1].set_ylabel('Total Funding (USD Billions)')
    axes[1].tick_params(axis='x', rotation=30)
    axes[1].grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig('../visualizations/success_rate_by_country.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()

## 20. Final Feature Summary Statistics

In [ ]:
print('=== FINAL FEATURE MATRIX STATISTICS ===')
summary_cols = [
    'Total_Funding_USD', 'Total_Funding_Rounds', 'Unique_Investors_Count',
    'Age_at_Latest_Round', 'Log_Total_Funding', 'Max_Investor_PageRank',
    'Sum_Investor_PageRank', 'Funding_Velocity'
]
summary_cols = [c for c in summary_cols if c in final_df.columns]
print(final_df[summary_cols].describe().round(4).to_string())
print(f'\n   Shape: {final_df.shape[0]:,} rows × {final_df.shape[1]} columns')
print(f'   Target distribution: {final_df["is_successful"].value_counts().to_dict()}')

## 21. Save Final ML-Ready Dataset

In [ ]:
os.makedirs('../data/processed', exist_ok=True)

output_path = '../data/processed/cleaned_data.csv'
final_df.to_csv(output_path, index=False)

print('='*60)
print('     FEATURE ENGINEERING COMPLETE!')
print('='*60)
print(f'\n✅ Final dataset saved to: {output_path}')
print(f'   Shape             : {final_df.shape[0]:,} rows × {final_df.shape[1]} columns')
print(f'   Total features    : {final_df.shape[1]}')
print(f'   Target (success)  : {final_df["is_successful"].sum():,} ({final_df["is_successful"].mean()*100:.2f}%)')
print('\n📌 Next steps:')
print('   → Run 05_Machine_Learning.ipynb  (XGBoost Classifier)')
print('   → Run 06_Forecasting.ipynb        (XGBoost Regressor)')

## 22. Generate Feature Engineering Report

In [ ]:
import datetime

report_path = '../reports/feature_engineering_report.txt'
os.makedirs('../reports', exist_ok=True)

with open(report_path, 'w') as f:
    f.write('='*70 + '\n')
    f.write('  STARTUP FUNDING ANALYSIS — FEATURE ENGINEERING REPORT\n')
    f.write(f'  Generated: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n')
    f.write('='*70 + '\n\n')

    f.write('DATASET SUMMARY\n')
    f.write('-'*40 + '\n')
    f.write(f'Input rows    : {len(df):,}\n')
    f.write(f'Output rows   : {len(final_df):,}\n')
    f.write(f'Total features: {final_df.shape[1]}\n\n')

    f.write('TARGET VARIABLE\n')
    f.write('-'*40 + '\n')
    f.write(f'Successful (1): {final_df["is_successful"].sum():,} ({final_df["is_successful"].mean()*100:.2f}%)\n')
    f.write(f'Not Successful(0): {(final_df["is_successful"]==0).sum():,}\n\n')

    f.write('INVESTOR NETWORK STATISTICS\n')
    f.write('-'*40 + '\n')
    f.write(f'Graph nodes (investors): {G.number_of_nodes():,}\n')
    f.write(f'Graph edges (syndicates): {G.number_of_edges():,}\n')
    f.write(f'Median PageRank: {median_pagerank:.10f}\n\n')

    f.write('TOP 10 INVESTORS BY PAGERANK\n')
    f.write('-'*40 + '\n')
    for i, (name, score) in enumerate(top5[:10], 1):
        f.write(f'{i:2d}. {name[:40]:40s}: {score:.10f}\n')

    f.write('\nFEATURE LIST\n')
    f.write('-'*40 + '\n')
    for col in final_df.columns:
        f.write(f'  - {col}\n')

print(f'✅ Feature engineering report saved: {report_path}')